# Comparing Feature Scaling Techniques on FIFA World Cup Data

Feature scaling is the process of transforming numeric features so they exist on a comparable scale. Many machine learning algorithms (distance-based models, gradient-based models, PCA) are sensitive to the raw magnitude of features. Without scaling, a feature like `attendance` (values in the tens of thousands) would dominate a feature like `total_teams` (values like 16-32) purely because of its larger numeric range, not because it's actually more important.

In this notebook, I compare three scaling techniques on real World Cup data:

- **Standard Scaling** - centers data to mean 0, spread to standard deviation 1
- **Robust Scaling** - centers data to median 0, spread to IQR 1 (designed to resist outliers)
- **Min-Max Scaling** - bounds data into a fixed 0 to 1 range

For each technique, I look at the formula, apply it to real data, verify the result numerically, and visualize the effect - especially how each technique handles an outlier present in the `attendance` column.

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

df = pd.read_csv("https://raw.githubusercontent.com/awais-DS/Data/refs/heads/main/FIFA_World_Cup_Champions_1930_2022.csv")
df.head()

## 2. Feature Selection

I selected `attendance` and `total_teams` as input features, and `top_scorer_goals` as the target. I deliberately excluded `year` - while it is numeric, scaling it doesn't carry a meaningful interpretation. A standardized year value doesn't represent anything a reader could naturally interpret; it's better treated as a timeline marker than a feature to be scaled.

In [ ]:
x = df[["attendance", "total_teams"]]
y = df["top_scorer_goals"]
x.head()

## 3. Train-Test Split

The scaler must be fit only on training data, then applied to test data using those same learned parameters. Fitting on the full dataset (or on test data) would leak information from the test set into training, giving an unrealistically optimistic view of model performance. This is why the split happens **before** scaling.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)
x_train.shape, x_test.shape

## 4. Standard Scaling

**Formula:** z = (x - mean) / standard deviation

This shifts the data so its mean becomes 0, and rescales it so its standard deviation becomes 1. The shape of the distribution is unaffected - only its position and spread change.

In [ ]:
standard_scaler = StandardScaler()
x_train_standard_scaled = standard_scaler.fit_transform(x_train)
x_test_standard_scaled = standard_scaler.transform(x_test)

x_train_standard_scaled = pd.DataFrame(x_train_standard_scaled, columns=x_train.columns)
x_test_standard_scaled = pd.DataFrame(x_test_standard_scaled, columns=x_test.columns)

x_train_standard_scaled.describe().round(2)

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

ax1.scatter(x_train['attendance'], x_train['total_teams'])
ax1.set_title("Before Scaling")
ax1.set_xlabel("Attendance")
ax1.set_ylabel("Total Teams")

ax2.scatter(x_train_standard_scaled['attendance'], x_train_standard_scaled['total_teams'], color='red')
ax2.set_title("After Standard Scaling")
ax2.set_xlabel("Attendance (scaled)")
ax2.set_ylabel("Total Teams (scaled)")

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

ax1.set_title('Before Scaling')
sns.kdeplot(x_train['attendance'], ax=ax1, label='attendance')
sns.kdeplot(x_train['total_teams'], ax=ax1, label='total_teams')
ax1.legend()

ax2.set_title('After Standard Scaling')
sns.kdeplot(x_train_standard_scaled['attendance'], ax=ax2, label='attendance')
sns.kdeplot(x_train_standard_scaled['total_teams'], ax=ax2, label='total_teams')
ax2.legend()

plt.tight_layout()
plt.show()

**Observation:** The `attendance` column contains one clear outlier - a World Cup with unusually high attendance compared to the rest. After standard scaling, this row lands at roughly **3.14 standard deviations** above the mean, clearly separated from the rest of the data, which clusters much closer to 0.

## 5. Robust Scaling

**Formula:** x' = (x - median) / IQR

Robust scaling belongs to the same family as standard scaling - it centers the data and rescales its spread. The difference is that it uses the median and interquartile range (IQR) instead of the mean and standard deviation, because median and IQR are not pulled around by extreme values. This does not bound the output to any fixed range - it remains unbounded, exactly like standard scaling.

In [ ]:
robust_scaler = RobustScaler()
x_train_robust_scaled = robust_scaler.fit_transform(x_train)
x_test_robust_scaled = robust_scaler.transform(x_test)

x_train_robust_scaled = pd.DataFrame(x_train_robust_scaled, columns=x_train.columns)
x_test_robust_scaled = pd.DataFrame(x_test_robust_scaled, columns=x_test.columns)

x_train_robust_scaled.describe().round(2)

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

ax1.scatter(x_train['attendance'], x_train['total_teams'])
ax1.axvline(x=robust_scaler.center_[0], color='red', linestyle='--', label='Median')
ax1.axhline(y=robust_scaler.center_[1], color='red', linestyle='--')
ax1.set_title("Before Scaling")
ax1.set_xlabel("Attendance")
ax1.set_ylabel("Total Teams")
ax1.legend()

ax2.scatter(x_train_robust_scaled['attendance'], x_train_robust_scaled['total_teams'], color='green')
ax2.axvline(x=0, color='red', linestyle='--', label='Median')
ax2.axhline(y=0, color='red', linestyle='--')
ax2.set_title("After Robust Scaling")
ax2.set_xlabel("Attendance (scaled)")
ax2.set_ylabel("Total Teams (scaled)")
ax2.legend()

plt.tight_layout()
plt.show()

**Observation:** Under robust scaling, the same outlier row reaches roughly **4.49** - even more extreme than the 3.14 seen with standard scaling. This is expected: IQR is smaller than standard deviation, so dividing by it stretches the outlier's distance from the center further. Robust scaling does not shrink or hide outliers - it protects the *normal* majority of the data from being distorted by them, while leaving the outlier clearly visible.

## 6. Min-Max Scaling

**Formula:** x' = (x - min) / (max - min)

Min-max scaling forces every value into a fixed 0 to 1 range. This works well when data has no extreme outliers, but it is highly sensitive to them: since the formula directly uses the minimum and maximum values, a single outlier can compress the rest of the data into a narrow band near one end of the range.

In [ ]:
minmax_scaler = MinMaxScaler()
x_train_minmax_scaled = minmax_scaler.fit_transform(x_train)
x_test_minmax_scaled = minmax_scaler.transform(x_test)

x_train_minmax_scaled = pd.DataFrame(x_train_minmax_scaled, columns=x_train.columns)
x_test_minmax_scaled = pd.DataFrame(x_test_minmax_scaled, columns=x_test.columns)

x_train_minmax_scaled.describe().round(2)

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

ax1.scatter(x_train['attendance'], x_train['total_teams'])
ax1.set_title("Before Scaling")
ax1.set_xlabel("Attendance")
ax1.set_ylabel("Total Teams")

ax2.scatter(x_train_minmax_scaled['attendance'], x_train_minmax_scaled['total_teams'], color='purple')
ax2.set_title("After Min-Max Scaling")
ax2.set_xlabel("Attendance (scaled)")
ax2.set_ylabel("Total Teams (scaled)")

plt.tight_layout()
plt.show()

**Observation:** The `attendance` column's 25th, 50th, and 75th percentiles all land between roughly **0.18 and 0.35** - the bulk of the "normal" data is squeezed into a tight lower band. The single outlier alone occupies the value 1.0, leaving a large empty gap between the normal cluster and the outlier. This is the clearest evidence in this notebook of why min-max scaling should be used cautiously on outlier-heavy data.

## 7. Summary

| Technique | Centers using | Scales using | Bounded range | Outlier behavior |
|---|---|---|---|---|
| Standard Scaling | Mean | Standard deviation | No (unbounded) | Outlier inflates SD, mildly distorting scale for all points |
| Robust Scaling | Median | IQR | No (unbounded) | Outlier is exaggerated, but normal data stays undistorted |
| Min-Max Scaling | Min | Min-Max range | Yes (0 to 1) | Outlier compresses all normal data into a narrow band |

**When to use which:**

- **Standard scaling** - good default for algorithms that assume roughly Gaussian-shaped input (linear regression, logistic regression, PCA), when the data doesn't have severe outliers.
- **Robust scaling** - preferred when the dataset has meaningful outliers that shouldn't be allowed to distort the scale for the rest of the data, such as this `attendance` column.
- **Min-max scaling** - useful when a fixed range is required (e.g. image pixel data, neural network inputs expecting 0-1), and the data has no significant outliers.

On this dataset specifically, `attendance` has a genuine outlier - a World Cup with unusually high attendance. Robust scaling handled this most cleanly, keeping the rest of the data well-distributed while still allowing the outlier to be identifiable rather than hidden.